# Prompt Flows in RAG — Bramblewood Gear Co. / FieldWise

**Company:** Bramblewood Gear Co., an outdoor equipment retailer.
**Product:** FieldWise, the internal assistant support agents use to answer
gear-care, troubleshooting, and warranty questions.

This notebook covers three prompt-flow patterns that sit between retrieval
and generation:

- Query rewriting (turning a vague ticket into a real search query)
- Contextual compression (trimming a long care article down to what matters)
- Structured prompt templates (one template, reused for every ticket, with
  a tone that can flex between a quick internal note and a customer-facing
  reply)

## Learning goals

By the end of this notebook, you should be able to:

1. Rewrite a user question before retrieval.
2. Compress retrieved context into a smaller, cleaner prompt.
3. Build reusable prompt templates for retrieval augmentation.
4. See how these ideas fit into hybrid RAG.

## 1) Install packages

```bash
pip install -U scikit-learn langchain langchain-community langchain-text-splitters langchain-core faiss-cpu
```

If you later want to run this with a live embedding model, also install
`sentence-transformers` and `langchain-huggingface`.

In [ ]:
%pip install -qU scikit-learn langchain langchain-community langchain-text-splitters langchain-core faiss-cpu

## 2) Build a small FieldWise knowledge base

Four short gear-care and policy passages -- the same kind of content a
support agent would actually need FieldWise to search over.

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=(
            "Tent rainflies use a factory-applied waterproof coating that "
            "wears down with UV exposure. If a rainfly starts leaking, "
            "reapply a DWR (durable water repellent) treatment and reseal "
            "the seams with seam sealer."
        ),
        metadata={"source": "kb_1_tent_waterproofing", "topic": "tents"},
    ),
    Document(
        page_content=(
            "Returns are accepted within 60 days for unused gear in original "
            "packaging. Used gear can only be handled through a warranty "
            "claim: 2 years on tents and stoves, lifetime on backpack "
            "stitching."
        ),
        metadata={"source": "kb_2_returns_warranty", "topic": "policy"},
    ),
    Document(
        page_content=(
            "The Ember Down sleeping bag loses loft when washed with regular "
            "detergent, which strips its water-resistant DriDown treatment. "
            "Wash only with a technical down soap on a gentle cycle, then "
            "tumble dry on low heat with a few clean tennis balls to break up "
            "clumps and restore loft. Repeat drying if the bag still feels "
            "lumpy after the first cycle."
        ),
        metadata={"source": "kb_3_sleeping_bag_care", "topic": "sleeping_bags"},
    ),
    Document(
        page_content=(
            "Basecamp backpack zipper sliders occasionally stick or fail "
            "after heavy use. A dry silicone lubricant usually frees a stiff "
            "slider; a fully broken slider is covered under the lifetime "
            "workmanship warranty and can be swapped at no cost."
        ),
        metadata={"source": "kb_4_backpack_zippers", "topic": "backpacks"},
    ),
]

docs

## 3) Query rewriting

Support tickets rarely use the same words as the knowledge base. A rewrite
step keeps the agent's intent but makes the retrieval query more explicit.

In [ ]:
def rewrite_question(question: str) -> str:
    q = question.strip().lower()
    rules = [
        ("how do i fix it", "how do i troubleshoot and repair the product"),
        ("leaking", "waterproofing failure and seam sealing"),
        ("warranty", "warranty coverage and the claims process"),
        ("return", "return and warranty policy"),
        ("not fluffy anymore", "loss of loft in a down sleeping bag"),
        ("zipper", "zipper slider failure and lubrication"),
    ]
    rewritten = q
    for old, new in rules:
        if old in rewritten:
            rewritten = rewritten.replace(old, new)
    return rewritten

sample_question = "It keeps leaking, how do I fix it?"
print("Original :", sample_question)
print("Rewritten:", rewrite_question(sample_question))

## 4) Use the rewritten question for retrieval

The rewrite is not the final answer -- it is the search query used to fetch
better context. This is also where we check for a live embedding model and
fall back to an offline one if it isn't reachable (see the note in the code
cell below).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer


class OfflineHashingEmbeddings(Embeddings):
    """Deterministic, dependency-free embedding fallback used when a live
    sentence-transformer can't be loaded (no package, or no network access
    to huggingface.co). Stopwords are filtered so shared connector words
    ("the", "a", "would") don't dominate similarity over shared content
    words ("waterproofing", "warranty")."""

    def __init__(self, n_features: int = 512):
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        print("Embeddings: live sentence-transformers model loaded (semantic).")
        return candidate, "live"
    except Exception as e:
        print(f"Embeddings: falling back to offline hashing embeddings -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineHashingEmbeddings(), "offline"


embeddings, EMBEDDINGS_MODE = get_embeddings()

splitter = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=20)
chunks = splitter.split_documents(docs)

store = FAISS.from_documents(chunks, embeddings)

query = rewrite_question(sample_question)
retrieved = store.similarity_search(query, k=3)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Retrieved {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 5) Contextual compression

Raw retrieved chunks can be longer or noisier than the model needs.
Compression keeps the useful part and trims the rest.

In [ ]:
def compress_context(retrieved_docs, max_chars_per_doc: int = 120):
    compressed = []
    for doc in retrieved_docs:
        text = doc.page_content.strip()
        if len(text) > max_chars_per_doc:
            text = text[:max_chars_per_doc].rsplit(" ", 1)[0] + "..."
        compressed.append(Document(page_content=text, metadata=doc.metadata))
    return compressed

compressed_docs = compress_context(retrieved, max_chars_per_doc=80)

for i, doc in enumerate(compressed_docs, 1):
    print(f"--- Compressed {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 6) Build a context block

A compact context block is easier for a model (or a human agent skimming
FieldWise's suggestion) to use than a long raw dump of retrieved text.

In [ ]:
def build_context_block(docs):
    lines = []
    for i, doc in enumerate(docs, 1):
        src = doc.metadata.get("source", "unknown")
        lines.append(f"[{i}] source={src}: {doc.page_content}")
    return "\n".join(lines)

context_block = build_context_block(compressed_docs)
print(context_block)

## 7) Structured prompt templates

A template separates the fixed instructions from the parts that change per
ticket (question, context), which is what makes the same prompt reusable
across thousands of tickets.

In [ ]:
retrieval_prompt_template = """
You are FieldWise, Bramblewood's support assistant, answering only from the
provided context.

Question:
{question}

Context:
{context}

Instructions:
- Use only the context.
- If the context is incomplete, say so.
- Keep the answer short and direct.
"""

filled_prompt = retrieval_prompt_template.format(
    question=sample_question,
    context=context_block,
)

print(filled_prompt)

## 8) Why templates help in RAG

Templates make the retrieval prompt repeatable. They matter here because
the same ticket can need two different tones: a quick internal note for the
agent, or a fuller reply that goes straight to the customer.

In [ ]:
def make_retrieval_prompt(question: str, context: str, style: str = "concise") -> str:
    template = """
You are a {style} Bramblewood support assistant.

Question:
{question}

Context:
{context}

Answer using only the context.
"""
    return template.format(style=style, question=question, context=context)

print(make_retrieval_prompt(sample_question, context_block, style="concise"))
print("\n" + "=" * 60 + "\n")
print(make_retrieval_prompt(sample_question, context_block, style="warm, customer-facing"))

## 9) Basic prompt flow for RAG

A simple prompt flow often looks like this:

1. Rewrite the question.
2. Retrieve relevant chunks.
3. Compress the retrieved context.
4. Format the final prompt with a template.
5. Send it to the model.

In [ ]:
def rag_prompt_flow(question: str, vector_store, style: str = "concise"):
    rewritten = rewrite_question(question)
    retrieved_docs = vector_store.similarity_search(rewritten, k=3)
    compressed = compress_context(retrieved_docs, max_chars_per_doc=80)
    context = build_context_block(compressed)
    prompt = make_retrieval_prompt(rewritten, context, style=style)
    return {
        "original_question": question,
        "rewritten_question": rewritten,
        "retrieved_count": len(retrieved_docs),
        "compressed_count": len(compressed),
        "prompt": prompt,
    }

flow_result = rag_prompt_flow("It keeps leaking, how do I fix it?", store)
flow_result

## 10) Add a simple validation step

Before a reply goes out, it's worth checking whether the retrieved context
actually covers the terms the question needs.

In [ ]:
def validate_context(context: str, required_terms=None):
    required_terms = required_terms or []
    lower = context.lower()
    return all(term.lower() in lower for term in required_terms)

print(validate_context(context_block, required_terms=["waterproofing"]))
print(validate_context(context_block, required_terms=["billing"]))

## 11) Compare the three pieces

- Query rewriting improves what you search for.
- Context compression improves what you send to the model.
- Prompt templates improve how you format the final request, including tone.

Together, they form a clean prompt-flow layer on top of FieldWise's
retrieval.

## Key takeaways

- Rewrite vague ticket language before retrieval.
- Compress retrieved text so the prompt stays small and relevant.
- Use prompt templates to keep retrieval prompts reusable, and let a style
  parameter flex tone between internal and customer-facing replies.
- These are lightweight building blocks for hybrid RAG and more advanced
  agentic workflows, including the MCP-based agent later in this module.

## References

- Rewrite question: https://docs.langchain.com/oss/python/langgraph/agentic-rag#5-rewrite-question
- Hybrid RAG: https://docs.langchain.com/oss/python/langchain/retrieval#hybrid-rag
- Context compression: https://docs.langchain.com/oss/python/deepagents/context-engineering#context-compression
- Prompts vs. prompt templates: https://docs.langchain.com/langsmith/prompt-engineering-concepts#prompts-vs-prompt-templates
- Prompt template format: https://docs.langchain.com/langsmith/prompt-template-format

## Verification summary

| Section | Status | Notes |
|---|---|---|
| Package install | Executed | pip install cell |
| Knowledge base | Executed & verified | in-memory documents |
| Query rewriting | Executed & verified | pure string rules, deterministic |
| Embeddings + retrieval | Executed & verified | see printed `EMBEDDINGS_MODE` -- offline hashing fallback in a sandbox with no Hugging Face access, live sentence-transformers on a machine with internet |
| Compression / templates / validation | Executed & verified | pure Python, no external API calls |

This notebook has no Groq or LangSmith dependency, so everything above runs
identically whether or not those services are reachable.